In [1]:
#import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.weightstats import ttest_ind 
from statsmodels.distributions.empirical_distribution import ECDF

In [2]:
#set up
# sns.set(style='whitegrid')

In [3]:
#load dataset
df = pd.read_csv('data/customer_data.csv')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 12 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   id                  100000 non-null  int64 
 1   age                 100000 non-null  int64 
 2   gender              100000 non-null  object
 3   income              100000 non-null  int64 
 4   education           100000 non-null  object
 5   region              100000 non-null  object
 6   loyalty_status      100000 non-null  object
 7   purchase_frequency  100000 non-null  object
 8   purchase_amount     100000 non-null  int64 
 9   product_category    100000 non-null  object
 10  promotion_usage     100000 non-null  int64 
 11  satisfaction_score  100000 non-null  int64 
dtypes: int64(6), object(6)
memory usage: 9.2+ MB


In [5]:
df.head()

,id,age,gender,income,education,region,loyalty_status,purchase_frequency,purchase_amount,product_category,promotion_usage,satisfaction_score
0,1,27,Male,40682,Bachelor,East,Gold,frequent,18249,Books,0,6
1,2,29,Male,15317,Masters,West,Regular,rare,4557,Clothing,1,6
2,3,37,Male,38849,Bachelor,West,Silver,rare,11822,Clothing,0,6
3,4,30,Male,11568,HighSchool,South,Regular,frequent,4098,Food,0,7
4,5,31,Female,46952,College,North,Regular,occasional,19685,Clothing,1,5


In [6]:
df['purchase_amount'] = df['purchase_amount'].astype('float').round(2)
df.head()

,id,age,gender,income,education,region,loyalty_status,purchase_frequency,purchase_amount,product_category,promotion_usage,satisfaction_score
0,1,27,Male,40682,Bachelor,East,Gold,frequent,18249.0,Books,0,6
1,2,29,Male,15317,Masters,West,Regular,rare,4557.0,Clothing,1,6
2,3,37,Male,38849,Bachelor,West,Silver,rare,11822.0,Clothing,0,6
3,4,30,Male,11568,HighSchool,South,Regular,frequent,4098.0,Food,0,7
4,5,31,Female,46952,College,North,Regular,occasional,19685.0,Clothing,1,5


In [7]:
#decriptive statistic
desc_stats = df[['income', 'purchase_amount', 'satisfaction_score']].describe()
desc_stats

,income,purchase_amount,satisfaction_score
count,100000.000000,100000.000000,100000.000000
mean,27516.269880,9634.790840,5.009650
std,12996.782587,4799.339449,1.038714
min,5000.000000,1118.000000,0.000000
25%,16271.750000,5583.000000,4.000000
50%,27584.500000,9452.000000,5.000000
75%,38747.250000,13350.000000,6.000000
max,50000.000000,26204.000000,10.000000


In [8]:
male_purchase = df[df['gender'] == 'Male']['purchase_amount']
female_purchase = df[df['gender'] == 'Female']['purchase_amount']
t_stat, p_value, _ = ttest_ind(male_purchase, female_purchase, usevar='unequal')
print(t_stat)
print(p_value)

0.025486774705113416
0.979666748175195


In [19]:
#confidence interval for satisfaction score of highest income quartile
q3_income = df['income'].quantile(0.75)
highest_income = df[df['income'] >= q3_income]['satisfaction_score']
ci_mean = highest_income.mean()
ci_std = highest_income.std()
ci_len = len(highest_income)
ci95 = stats.t.interval(0.95, ci_len - 1, loc=ci_mean, scale=ci_std/np.sqrt(ci_len))

In [20]:
#correlation matrix
corr = df[['income', 'purchase_amount', 'satisfaction_score']].corr()
corr

,income,purchase_amount,satisfaction_score
income,1.000000,0.948441,0.002780
purchase_amount,0.948441,1.000000,0.003424
satisfaction_score,0.002780,0.003424,1.000000


In [28]:
#probability distribution
k_stat_norm, k_pvalue_norm = stats.kstest(
    df['purchase_amount'], 
    'norm', 
    args=(
        df['purchase_amount'].mean(), 
        df['purchase_amount'].std()
    )
)
k_stat_expo, k_pvalue_expo = stats.kstest(
    df['purchase_amount'],
    'expon',
    args=(
        df['purchase_amount'].min(),
        df['purchase_amount'].mean()
    )
)
print(k_stat_norm)
print(k_pvalue_norm)
print(k_stat_expo)
print(k_pvalue_expo)

0.053063977700982756
3.604849145928401e-245
0.1340804529941544
0.0


In [51]:
#outlier detection
z_scores = np.abs(stats.zscore(df[['purchase_amount', 'satisfaction_score']]))
outliers = (z_scores > 3).any(axis=1)
outlier_df = df[outliers]
# outlier_df.head(70)
# outlier_len = df[df['outlier'] == True]
# outlier_lenn = len(outlier_len)
# outlier_lenn

In [54]:
#promotion_usage vs statisfaction
promotion_usage = df.groupby('promotion_usage')['satisfaction_score'].mean()
promotion_usage

promotion_usage
0    5.009425
1    5.010173
Name: satisfaction_score, dtype: float64